# Meezan Bank Branch Location Graph

**Goal:** corrected CSV → validation → normalized records → Neo4j graph → spatial properties → graph validation.

Graph structure: `Branch → Area → City → Province`.

**Design rules**
- Branch identity is `(dataset_id, branch_code)`.
- City identity includes province; Area identity includes province and city.
- Coordinates remain empty until geocoding supplies defensible results.
- Neo4j stores a native WGS-84 `POINT` for distance calculations.
- Ingestion uses stable keys and `MERGE`; this notebook does not wipe data unless the explicit reset switch is enabled.

## 1. Setup

In [3]:
from pathlib import Path
import os, re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from neo4j import GraphDatabase
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'notebooks').exists():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks/ folder.')

DATASET_ID = 'meezan_branches'

XLSX_PATH = PROJECT_ROOT / 'data' / 'raw' / 'meezan.xlsx'
EXPECTED_BRANCH_COUNT = 1102

# False preserves existing data and MERGEs. True deletes ONLY this dataset before rebuilding.
RESET_DATASET_BEFORE_INGEST = False

env_path = find_dotenv(usecwd=True) or str(PROJECT_ROOT / '.env')
if not Path(env_path).exists():
    raise FileNotFoundError('Could not find the project .env file.')
load_dotenv(env_path, override=True)

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
missing = [k for k, v in {'NEO4J_URI': NEO4J_URI, 'NEO4J_USERNAME': NEO4J_USERNAME,
                          'NEO4J_PASSWORD': NEO4J_PASSWORD}.items() if not v]
if missing:
    raise ValueError('Missing environment variables: ' + ', '.join(missing))

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Loaded .env:', env_path)
print('Project root:', PROJECT_ROOT)
print('XLSX:', XLSX_PATH)
print('Neo4j connection verified; credentials hidden.')

Loaded .env: /home/ekkel-ai/momina/projects/graphrag/.env
Project root: /home/ekkel-ai/momina/projects/graphrag
XLSX: /home/ekkel-ai/momina/projects/graphrag/data/raw/meezan.xlsx
Neo4j connection verified; credentials hidden.


## 2. Load the meezan branch file

In [4]:
if not XLSX_PATH.exists():
    raise FileNotFoundError(f'XLSX not found: {XLSX_PATH}. Update XLSX_PATH in Section 1.')

df = pd.read_excel(XLSX_PATH, sheet_name="Sheet1", header=1, dtype=str)
df.columns = [str(c).strip() for c in df.columns]
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

required_columns = [
    'S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No',
    'Branch Landline Number', 'City Name', 'Area Name', 'Province', 'Branch Address'
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f'CSV is missing required columns: {missing_columns}')

print('Rows:', len(df))
print('Columns:', df.columns.tolist())
display(df.head())

Rows: 1102
Columns: ['S.No', 'Branch Code', 'Branch Name', 'OM Landline No', 'BM Landline No', 'Branch Landline Number', 'City Name', 'Area Name', 'Province', 'Branch Address']


,S.No,Branch Code,Branch Name,OM Landline No,BM Landline No,Branch Landline Number,City Name,Area Name,Province,Branch Address
0,1,105,"Marriott Hotel, Karachi",021-35693166,021-5682682,021-5683491,Karachi,Clifton,Sindh,"Marriott Hotel, Abdullah Haroon Road, Karachi"
1,2,201,"Gulberg, Lahore",042-35755716\n(042-35777434),042-35870929,042-5879870-72,Lahore,Gulberg,Punjab,"60 Main Boulevard, Gulberg, Lahore"
2,3,301,"Jinnah Avenue (Blue Area), Islamabad",051-2206513,051-2277518,051-2276712-15,Islamabad,Islamabad East,Islamabad,"Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ..."
3,4,101,"PNSC (Lalazar), Karachi",021-35636253,021-35636252,021-35636404,Karachi,Chundrigar Road,Sindh,"37-A, Lalazar Area, Off Moulvi Tamizuddin Khan..."
4,5,401,"Sereena Hotel, Faisalabad",041-2602594,041-2601225,041-2602595-97,Faisalabad,Commercial FSD,Punjab,"Shop No.G-737 to G-740, G-751 & G-753, Bazar C..."


## 3. Data validation

In [6]:
location_cols = ['City Name', 'Area Name', 'Province', 'Branch Address']
empty_codes = df[df['Branch Code'].str.strip() == '']
duplicate_codes = df[df['Branch Code'].duplicated(keep=False) & (df['Branch Code'] != '')]
missing_location = {c: int((df[c].str.strip() == '').sum()) for c in location_cols}
branch_3601 = df[df['Branch Code'].str.strip() == '3601']

print('Rows:', len(df), '| expected:', EXPECTED_BRANCH_COUNT)
print('Empty branch codes:', len(empty_codes))
print('Duplicate branch-code rows:', len(duplicate_codes))
print('Missing location values:', missing_location)
print('Branch 3601:')
display(branch_3601)
if len(duplicate_codes): display(duplicate_codes)

assert len(df) == EXPECTED_BRANCH_COUNT, f'Expected {EXPECTED_BRANCH_COUNT} rows; got {len(df)}.'
assert len(empty_codes) == 0, 'Empty branch codes found.'
assert len(duplicate_codes) == 0, 'Duplicate branch codes found.'
assert len(branch_3601) == 1, 'Expected exactly one branch 3601 row.'
assert branch_3601.iloc[0]['City Name'].strip().casefold() == 'hub', 'Branch 3601 city must be Hub.'
assert branch_3601.iloc[0]['Province'].strip().casefold() == 'balochistan', 'Branch 3601 province must be Balochistan.'
print('Validation passed.')

Rows: 1102 | expected: 1102
Empty branch codes: 0
Duplicate branch-code rows: 0
Missing location values: {'City Name': 0, 'Area Name': 0, 'Province': 0, 'Branch Address': 0}
Branch 3601:


,S.No,Branch Code,Branch Name,OM Landline No,BM Landline No,Branch Landline Number,City Name,Area Name,Province,Branch Address
133,134,3601,Hub Chowki Lasbela Branch,0853-310251,0853-310252,0853-363312-16,Hub,SITE,Balochistan,"Plot Bearing No. 221/222, Mouza Berot, Tehsil ..."


Validation passed.


## 4. Data normalization and identity keys

- Province: `province`
- City: `province|city`
- Area: `province|city|area`
- Branch: `(dataset_id, branch_code)`

This prevents same-named cities or areas in different provinces from being merged into one node.

In [8]:
def normalize_name(value):
    value = '' if value is None else str(value)
    return re.sub(r'\s+', ' ', value.strip().casefold())

def clean_value(value):
    """Convert missing or mixed-type CSV values safely into strings."""
    if pd.isna(value):
        return ""
    return str(value).strip()

records = []

for row in df.to_dict(orient="records"):

    branch_code = clean_value(row["Branch Code"])

    if not branch_code:
        continue

    city = clean_value(row["City Name"])
    area = clean_value(row["Area Name"])
    province = clean_value(row["Province"])

    province_key = normalize_name(province)
    city_name_key = normalize_name(city)
    area_name_key = normalize_name(area)

    records.append({
        "dataset_id": DATASET_ID,
        "branch_code": branch_code,
        "branch_key": f"{DATASET_ID}|{branch_code}",
        "branch_name": clean_value(row["Branch Name"]),

        "city": city,
        "city_key": f"{province_key}|{city_name_key}",

        "area": area,
        "area_key": f"{province_key}|{city_name_key}|{area_name_key}",

        "province": province,
        "province_key": province_key,

        "address": clean_value(row["Branch Address"]),
        "om_landline": clean_value(row["OM Landline No"]),
        "bm_landline": clean_value(row["BM Landline No"]),
        "branch_landline": clean_value(row["Branch Landline Number"]),
        "source_sno": clean_value(row["S.No"]),

        # Coordinates will be populated after geocoding.
        "latitude": None,
        "longitude": None,
        "geocoding_status": "pending",
        "geocoding_source": None,
        "matched_address": None,
        "location_precision": "unknown"
    })

records_df = pd.DataFrame(records)

print("Graph-ready records:", len(records))
print("Unique branch keys:", records_df["branch_key"].nunique())

display(records_df.head())

Graph-ready records: 1102
Unique branch keys: 1102


,dataset_id,branch_code,branch_key,branch_name,city,city_key,area,area_key,province,province_key,...,om_landline,bm_landline,branch_landline,source_sno,latitude,longitude,geocoding_status,geocoding_source,matched_address,location_precision
0,meezan_branches,105,meezan_branches|105,"Marriott Hotel, Karachi",Karachi,sindh|karachi,Clifton,sindh|karachi|clifton,Sindh,sindh,...,021-35693166,021-5682682,021-5683491,1,None,None,pending,None,None,unknown
1,meezan_branches,201,meezan_branches|201,"Gulberg, Lahore",Lahore,punjab|lahore,Gulberg,punjab|lahore|gulberg,Punjab,punjab,...,042-35755716\n(042-35777434),042-35870929,042-5879870-72,2,None,None,pending,None,None,unknown
2,meezan_branches,301,meezan_branches|301,"Jinnah Avenue (Blue Area), Islamabad",Islamabad,islamabad|islamabad,Islamabad East,islamabad|islamabad|islamabad east,Islamabad,islamabad,...,051-2206513,051-2277518,051-2276712-15,3,None,None,pending,None,None,unknown
3,meezan_branches,101,meezan_branches|101,"PNSC (Lalazar), Karachi",Karachi,sindh|karachi,Chundrigar Road,sindh|karachi|chundrigar road,Sindh,sindh,...,021-35636253,021-35636252,021-35636404,4,None,None,pending,None,None,unknown
4,meezan_branches,401,meezan_branches|401,"Sereena Hotel, Faisalabad",Faisalabad,punjab|faisalabad,Commercial FSD,punjab|faisalabad|commercial fsd,Punjab,punjab,...,041-2602594,041-2601225,041-2602595-97,5,None,None,pending,None,None,unknown


## 5. Preview before writing

Review the generated keys and branch 3601. No graph data is written in this section.

In [9]:
preview_cols = ['branch_code','branch_name','area','city','province','address','province_key','city_key','area_key']
display(records_df[preview_cols].head(15))
print('Corrected branch 3601:')
display(records_df[records_df['branch_code'] == '3601'][preview_cols])

assert records_df['branch_code'].is_unique
assert records_df['city_key'].notna().all()
assert records_df['area_key'].notna().all()

,branch_code,branch_name,area,city,province,address,province_key,city_key,area_key
0,105,"Marriott Hotel, Karachi",Clifton,Karachi,Sindh,"Marriott Hotel, Abdullah Haroon Road, Karachi",sindh,sindh|karachi,sindh|karachi|clifton
1,201,"Gulberg, Lahore",Gulberg,Lahore,Punjab,"60 Main Boulevard, Gulberg, Lahore",punjab,punjab|lahore,punjab|lahore|gulberg
2,301,"Jinnah Avenue (Blue Area), Islamabad",Islamabad East,Islamabad,Islamabad,"Plot # 37-B, Tahir Plaza, Jinnah Avenue, Blue ...",islamabad,islamabad|islamabad,islamabad|islamabad|islamabad east
3,101,"PNSC (Lalazar), Karachi",Chundrigar Road,Karachi,Sindh,"37-A, Lalazar Area, Off Moulvi Tamizuddin Khan...",sindh,sindh|karachi,sindh|karachi|chundrigar road
4,401,"Sereena Hotel, Faisalabad",Commercial FSD,Faisalabad,Punjab,"Shop No.G-737 to G-740, G-751 & G-753, Bazar C...",punjab,punjab|faisalabad,punjab|faisalabad|commercial fsd
5,104,"Gulshan-e-Iqbal, Karachi",Gulshan-e-Iqbal,Karachi,Sindh,"Shop No. 1, 2, 3 & 4, Plot No.B-41, Block 13-A...",sindh,sindh|karachi,sindh|karachi|gulshan-e-iqbal
6,402,"Kotwali Road, Faisalabad",City Faisalabad,Faisalabad,Punjab,"P-66 Allama Iqbal Road (Kotwali Road), Faisalabad",punjab,punjab|faisalabad,punjab|faisalabad|city faisalabad
7,501,Nawan Shaher Branch Multan (Main Branch),Ahmad Shah Abdali,Multan,Punjab,"92,93,94, LMQ Road, Chowk Nawan Shaher, Multan",punjab,punjab|multan,punjab|multan|ahmad shah abdali
8,102,"Jodia Bazar, Karachi",Jodia Bazar,Karachi,Sindh,"H 9/A Darya Lal Street, Market Quarters Jodia ...",sindh,sindh|karachi,sindh|karachi|jodia bazar
9,103,"F.T.C, Karachi",Shara-e-Faisal,Karachi,Sindh,"Ground Floor, Block-B, Finance & Trade Centre,...",sindh,sindh|karachi,sindh|karachi|shara-e-faisal


Corrected branch 3601:


,branch_code,branch_name,area,city,province,address,province_key,city_key,area_key
133,3601,Hub Chowki Lasbela Branch,SITE,Hub,Balochistan,"Plot Bearing No. 221/222, Mouza Berot, Tehsil ...",balochistan,balochistan|hub,balochistan|hub|site


## 6. Neo4j constraints and indexes

Composite uniqueness constraints enforce the same identity rules inside Neo4j. These statements are safe to rerun because they use `IF NOT EXISTS`.

In [10]:
constraint_statements = [
    'CREATE CONSTRAINT branch_dataset_code_unique IF NOT EXISTS FOR (b:Branch) REQUIRE (b.dataset_id, b.branch_code) IS UNIQUE',
    'CREATE CONSTRAINT province_dataset_key_unique IF NOT EXISTS FOR (p:Province) REQUIRE (p.dataset_id, p.name_key) IS UNIQUE',
    'CREATE CONSTRAINT city_dataset_key_unique IF NOT EXISTS FOR (c:City) REQUIRE (c.dataset_id, c.name_key) IS UNIQUE',
    'CREATE CONSTRAINT area_dataset_key_unique IF NOT EXISTS FOR (a:Area) REQUIRE (a.dataset_id, a.name_key) IS UNIQUE',
]
with driver.session() as session:
    for statement in constraint_statements:
        session.run(statement).consume()
    session.run('CREATE INDEX branch_code_lookup IF NOT EXISTS FOR (b:Branch) ON (b.branch_code)').consume()
print('Uniqueness constraints and branch lookup index are ready.')

Uniqueness constraints and branch lookup index are ready.


## 7. Graph ingestion

Create `Branch → Area → City → Province` using stable keys. The default reset switch is `False`, so this section preserves existing data and merges matching nodes. If you intentionally want a clean rebuild, set `RESET_DATASET_BEFORE_INGEST = True` in Section 1, restart the kernel, and run all cells again. The reset is restricted to nodes with this `dataset_id`.

In [11]:
if RESET_DATASET_BEFORE_INGEST:
    confirmation = input(f'Type DELETE {DATASET_ID} to delete only this dataset: ')
    if confirmation != f'DELETE {DATASET_ID}':
        raise RuntimeError('Reset cancelled; no graph data deleted.')
    with driver.session() as session:
        session.run('MATCH (n {dataset_id: $dataset_id}) DETACH DELETE n', dataset_id=DATASET_ID).consume()
    print('Deleted existing nodes for this dataset.')
else:
    print('Reset disabled: existing graph data will be preserved.')

upsert_query = '''
UNWIND $rows AS row
MERGE (p:Province {dataset_id: row.dataset_id, name_key: row.province_key})
SET p.name = row.province
MERGE (c:City {dataset_id: row.dataset_id, name_key: row.city_key})
SET c.name = row.city, c.province_key = row.province_key
MERGE (a:Area {dataset_id: row.dataset_id, name_key: row.area_key})
SET a.name = row.area, a.city_name = row.city,
    a.city_key = row.city_key, a.province_key = row.province_key
MERGE (b:Branch {dataset_id: row.dataset_id, branch_code: row.branch_code})
SET b.branch_name = row.branch_name,
    b.city = row.city, b.area = row.area, b.province = row.province,
    b.address = row.address, b.om_landline = row.om_landline,
    b.bm_landline = row.bm_landline, b.branch_landline = row.branch_landline,
    b.source_sno = row.source_sno,
    b.geocoding_status = coalesce(b.geocoding_status, row.geocoding_status)
MERGE (b)-[:LOCATED_IN]->(a)
MERGE (a)-[:WITHIN_CITY]->(c)
MERGE (c)-[:WITHIN_PROVINCE]->(p)
'''

BATCH_SIZE = 250
with driver.session() as session:
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start+BATCH_SIZE]
        session.run(upsert_query, rows=batch).consume()
        print(f'Ingested {min(start+BATCH_SIZE, len(records))}/{len(records)} rows')
print('Graph ingestion completed.')

Reset disabled: existing graph data will be preserved.
Ingested 250/1102 rows
Ingested 500/1102 rows
Ingested 750/1102 rows
Ingested 1000/1102 rows
Ingested 1102/1102 rows
Graph ingestion completed.


## 8. Spatial properties

Each geocoded branch will eventually have numeric `latitude` and `longitude`, a native Neo4j `location` point, and provenance fields such as geocoding source, matched address, and location precision.

In [12]:
with driver.session() as session:
    session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        SET b.geocoding_status = coalesce(b.geocoding_status, 'pending')
    ''', dataset_id=DATASET_ID).consume()

    result = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        WHERE b.latitude IS NOT NULL AND b.longitude IS NOT NULL
          AND toFloat(b.latitude) >= -90 AND toFloat(b.latitude) <= 90
          AND toFloat(b.longitude) >= -180 AND toFloat(b.longitude) <= 180
        SET b.latitude = toFloat(b.latitude),
            b.longitude = toFloat(b.longitude),
            b.location = point({latitude: toFloat(b.latitude), longitude: toFloat(b.longitude)})
        RETURN count(b) AS points_created
    ''', dataset_id=DATASET_ID).single()

print('Branches with valid spatial points:', result['points_created'])
with driver.session() as session:
    statuses = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        RETURN b.geocoding_status AS status, count(*) AS branches
        ORDER BY status
    ''', dataset_id=DATASET_ID).data()
display(pd.DataFrame(statuses))

Branches with valid spatial points: 0


,status,branches
0,pending,1102


## 9. Graph validation

Validate node counts, relationship counts, duplicate branch nodes, complete administrative paths, and the corrected branch 3601 route. The key invariant is **one complete Area → City → Province path per branch**.

In [13]:
with driver.session() as session:
    node_counts = session.run('''
        MATCH (n {dataset_id: $dataset_id})
        UNWIND labels(n) AS label
        RETURN label, count(*) AS total ORDER BY label
    ''', dataset_id=DATASET_ID).data()
    rel_counts = session.run('''
        MATCH (x {dataset_id: $dataset_id})-[r]->(y {dataset_id: $dataset_id})
        RETURN type(r) AS relationship, count(*) AS total ORDER BY relationship
    ''', dataset_id=DATASET_ID).data()
    path_summary = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        OPTIONAL MATCH (b)-[:LOCATED_IN]->(a:Area)
        OPTIONAL MATCH (a)-[:WITHIN_CITY]->(c:City)
        OPTIONAL MATCH (c)-[:WITHIN_PROVINCE]->(p:Province)
        RETURN count(DISTINCT b) AS branches,
               count(DISTINCT CASE WHEN a IS NOT NULL THEN b END) AS with_area,
               count(DISTINCT CASE WHEN c IS NOT NULL THEN b END) AS with_city,
               count(DISTINCT CASE WHEN p IS NOT NULL THEN b END) AS with_province
    ''', dataset_id=DATASET_ID).single()
    duplicate_nodes = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        WITH b.branch_code AS code, count(*) AS copies
        WHERE copies > 1 RETURN code, copies ORDER BY code
    ''', dataset_id=DATASET_ID).data()
    bad_paths = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id})
        OPTIONAL MATCH (b)-[:LOCATED_IN]->(a:Area)
        OPTIONAL MATCH (a)-[:WITHIN_CITY]->(c:City)
        OPTIONAL MATCH (c)-[:WITHIN_PROVINCE]->(p:Province)
        WITH b, count(DISTINCT a) AS areas, count(DISTINCT c) AS cities,
             count(DISTINCT p) AS provinces
        WHERE areas <> 1 OR cities <> 1 OR provinces <> 1
        RETURN b.branch_code AS branch_code, areas, cities, provinces
        ORDER BY branch_code
    ''', dataset_id=DATASET_ID).data()

print('NODE COUNTS'); display(pd.DataFrame(node_counts))
print('RELATIONSHIP COUNTS'); display(pd.DataFrame(rel_counts))
print('PATH SUMMARY'); display(pd.DataFrame([dict(path_summary)]))
print('DUPLICATE BRANCH NODES'); display(pd.DataFrame(duplicate_nodes))
print('BRANCHES WITH ZERO OR MULTIPLE PATHS'); display(pd.DataFrame(bad_paths))

assert path_summary['branches'] == EXPECTED_BRANCH_COUNT
assert path_summary['with_area'] == EXPECTED_BRANCH_COUNT
assert path_summary['with_city'] == EXPECTED_BRANCH_COUNT
assert path_summary['with_province'] == EXPECTED_BRANCH_COUNT
assert not duplicate_nodes, 'Duplicate Branch nodes found.'
assert not bad_paths, 'Some branches do not have exactly one complete administrative path.'
print('Graph validation passed.')

NODE COUNTS


,label,total
0,Area,397
1,Branch,1102
2,City,329
3,Province,7


RELATIONSHIP COUNTS


,relationship,total
0,LOCATED_IN,1102
1,WITHIN_CITY,397
2,WITHIN_PROVINCE,329


PATH SUMMARY


,branches,with_area,with_city,with_province
0,1102,1102,1102,1102


DUPLICATE BRANCH NODES


""


BRANCHES WITH ZERO OR MULTIPLE PATHS


""


Graph validation passed.


In [14]:
with driver.session() as session:
    result_3601 = session.run('''
        MATCH (b:Branch {dataset_id: $dataset_id, branch_code: '3601'})
              -[:LOCATED_IN]->(a:Area)
              -[:WITHIN_CITY]->(c:City)
              -[:WITHIN_PROVINCE]->(p:Province)
        RETURN b.branch_code AS branch_code, b.branch_name AS branch_name,
               b.city AS branch_city_property, b.province AS branch_province_property,
               a.name AS graph_area, c.name AS graph_city, p.name AS graph_province
    ''', dataset_id=DATASET_ID).data()
display(pd.DataFrame(result_3601))
assert len(result_3601) == 1
assert result_3601[0]['graph_city'].casefold() == 'hub'
assert result_3601[0]['graph_province'].casefold() == 'balochistan'
print('Branch 3601 is correctly connected to Hub, Balochistan.')

,branch_code,branch_name,branch_city_property,branch_province_property,graph_area,graph_city,graph_province
0,3601,Hub Chowki Lasbela Branch,Hub,Balochistan,SITE,Hub,Balochistan


Branch 3601 is correctly connected to Hub, Balochistan.


In [17]:
print("=" * 70)
print("MEEZAN BRANCHES DATABASE SCHEMA")
print("=" * 70)

with driver.session() as session:

    # 1. Inspect node labels, properties, and counts
    node_query = """
    MATCH (n)
    WHERE n.dataset_id = $dataset_id
    UNWIND labels(n) AS node_label
    RETURN
        node_label,
        collect(DISTINCT keys(n)) AS property_groups,
        count(DISTINCT n) AS node_count
    ORDER BY node_label
    """

    node_results = session.run(
        node_query,
        dataset_id=DATASET_ID
    )

    print("\nNODE LABELS AND PROPERTIES")
    print("-" * 70)

    for record in node_results:
        print(f"\nNode: {record['node_label']}")
        print(f"Count: {record['node_count']:,}")

        all_properties = set()
        for group in record["property_groups"]:
            all_properties.update(group)

        print("Properties:")
        for prop in sorted(all_properties):
            print(f"  - {prop}")

    # 2. Inspect relationships
    relationship_query = """
    MATCH (source)-[r]->(target)
    WHERE source.dataset_id = $dataset_id
      AND target.dataset_id = $dataset_id
    RETURN
        labels(source) AS source_labels,
        type(r) AS relationship,
        labels(target) AS target_labels,
        count(r) AS relationship_count
    ORDER BY relationship
    """

    relationship_results = session.run(
        relationship_query,
        dataset_id=DATASET_ID
    )

    print("\n\nRELATIONSHIPS")
    print("-" * 70)

    for record in relationship_results:
        source = ":".join(record["source_labels"])
        target = ":".join(record["target_labels"])

        print(
            f"(:{source}) -[:{record['relationship']}]-> "
            f"(:{target}) | Count: {record['relationship_count']:,}"
        )

print("\n" + "=" * 70)

MEEZAN BRANCHES DATABASE SCHEMA

NODE LABELS AND PROPERTIES
----------------------------------------------------------------------

Node: Area
Count: 397
Properties:
  - city_key
  - city_name
  - dataset_id
  - name
  - name_key
  - province_key

Node: Branch
Count: 1,102
Properties:
  - address
  - area
  - bm_landline
  - branch_code
  - branch_landline
  - branch_name
  - city
  - dataset_id
  - geocoding_status
  - om_landline
  - province
  - source_sno

Node: City
Count: 329
Properties:
  - dataset_id
  - name
  - name_key
  - province_key

Node: Province
Count: 7
Properties:
  - dataset_id
  - name
  - name_key


RELATIONSHIPS
----------------------------------------------------------------------
(:Branch) -[:LOCATED_IN]-> (:Area) | Count: 1,102
(:Area) -[:WITHIN_CITY]-> (:City) | Count: 397
(:City) -[:WITHIN_PROVINCE]-> (:Province) | Count: 329



## 10. Spatial query test

Set a real test coordinate after geocoding. `point.distance()` returns straight-line distance in metres; it is not road-network travel distance. Road distance requires a routing engine such as OSRM.

In [ ]:
TEST_LATITUDE = None   # e.g. 31.5204
TEST_LONGITUDE = None  # e.g. 74.3587
NEAREST_LIMIT = 5

if TEST_LATITUDE is None or TEST_LONGITUDE is None:
    print('Spatial test skipped: add a real latitude/longitude after geocoding.')
else:
    TEST_LATITUDE, TEST_LONGITUDE = float(TEST_LATITUDE), float(TEST_LONGITUDE)
    if not -90 <= TEST_LATITUDE <= 90: raise ValueError('Latitude must be between -90 and 90.')
    if not -180 <= TEST_LONGITUDE <= 180: raise ValueError('Longitude must be between -180 and 180.')
    query = '''
        MATCH (b:Branch {dataset_id: $dataset_id})
        WHERE b.location IS NOT NULL
        WITH b, point.distance(b.location, point({latitude: $latitude, longitude: $longitude})) AS distance_m
        RETURN b.branch_code AS branch_code, b.branch_name AS branch_name,
               b.address AS address, b.city AS city, b.area AS area, b.province AS province,
               round(distance_m / 1000.0, 2) AS distance_km
        ORDER BY distance_m LIMIT $limit
    '''
    with driver.session() as session:
        nearest = session.run(query, dataset_id=DATASET_ID,
                              latitude=TEST_LATITUDE, longitude=TEST_LONGITUDE,
                              limit=NEAREST_LIMIT).data()
    display(pd.DataFrame(nearest))

## Final notes and cleanup

- Keep the corrected CSV as the source of truth.
- Review a sample of geocoding matches before processing all branches.
- Store geocoding source, returned address, and precision; do not call a locality centroid an exact branch location.
- Rerun Sections 8–10 after coordinates are added.
- Close the Neo4j driver when finished.

In [ ]:
driver.close()
print('Neo4j driver closed.')